The contents of this notebook were created with assistance from Claude generative AI.

**Runtime:** this stage ran on **WSL2 (Ubuntu) under Windows**, using both NVIDIA RTX 4090 GPUs. The dual-GPU mode is built into this notebook — duplicate it, set `WORKER_MODE=True` with `WORKER_GPU="0"` and `"1"` in the two copies, and *Run All* on each; the copies coordinate through a shared on-disk claim queue. For a single GPU, leave `WORKER_MODE=False`.

# CBDTP Relevance + Stance — vLLM **v5** (v1 re-run: keep v1, fix its parse-failures)
### Milestone 2 — v5 = v1's labels ⊕ properly-processed v1 parse-failures

**Why v5:** against the hand-labeled gold, **v1** was the best vLLM relevance labeler (F1 86%, recall 77%, precision
97%) — its only real defect was **17.9% parse-failures** caused by its terse free-text output format, which silently
defaulted to off-topic. v5 keeps **every v1 `parsed=True` label verbatim** and **re-labels only v1's `parsed=False`
rows** using **v1's exact judgment prompt** but emitted as a guided single JSON OBJECT, so they can no longer fail
to parse. Net result = v1's judgment with the ~18% hole filled in.

1. **v1's prompt, verbatim judgment** — zero-shot relevance+stance instructions copied from the v1 notebook (the
   judgment that matched gold best). NOT the v2/v4 few-shot — we are reproducing v1, not changing it.
2. **Single JSON OBJECT output (A)** — one target per call, object schema, so an empty array `[]` is grammatically
   impossible under guided decoding: the format failure that sank v1 cannot recur.
3. **Seed + reprocess (no full relabel)** — §8d copies v1's good rows into the v5 namespace and reprocesses ONLY
   the v1 omissions (their specific comment ids per thread), so v1's kept labels are bit-for-bit unchanged.
4. **§E pre-routing + `fail_reason`** — no-signal targets routed deterministically; every reprocessed row carries
   a `fail_reason` so coverage is auditable (want ~0%).

All outputs carry the **`-vllm-v5`** suffix — v1/v2/v3/v4 untouched.

**Platform:** vLLM is Linux-only — run under **WSL2** or **native Linux**. This notebook will not import on
native Windows.

## 0 · Run mode — single GPU or both via workers

Always **one GPU per process** (tensor-parallel across the non-NVLink 4090s is slower and stalls — removed).

- **Single GPU:** leave `WORKER_MODE=False`, set `WORKER_GPU` to `"0"` or `"1"`, Run All. The main notebook
  classifies all comment threads → posts → merge on that one card.
- **Both GPUs (recommended for the full relabel):** the pre-built **`_dp0` / `_dp1`** copies have `WORKER_MODE=True`
  and `WORKER_GPU` pinned to `0` / `1`. Run All on each — they pull from a **shared claim queue** (§9b): no
  cross-GPU comm, no static split, no idle when one finishes early. Stop one with **Interrupt** (releases its
  claim; engine stays loaded → re-run §9b to resume). Do the final merge from the main notebook
  (`WORKER_MODE=False`).

`GPU_MEM_UTIL` (default `0.90`; `0.82` on display-GPU 0), `MAX_NUM_SEQS` (default `128`), `MAX_TARGETS_PER_CHUNK`
(default `1`) and `MADS_DATA_DIR` are env-overridable.

## 1 · Configuration

In [1]:
%%time
import os, re, json, time, hashlib, warnings, math
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import sys
try: sys.stdout.reconfigure(encoding="utf-8")   # robust prints regardless of console codepage
except Exception: pass

# --- GPU selection (Linux/vLLM): set CUDA_VISIBLE_DEVICES BEFORE importing vllm/torch. Always ONE GPU per
#     process (tensor-parallel across the non-NVLink 4090s is slower and stalls — never used).
# ══ GPU — pick the card ═════════════════════════════════════════════════════════════════════════════════
WORKER_MODE = False        # for the dynamic data-parallel workers: DUPLICATE this notebook, set WORKER_MODE=True in
                           # both copies and WORKER_GPU "0"/"1", Run All on each. They share one claim queue.
WORKER_GPU  = "1"          # "0" or "1" — the GPU this process runs on (set differently in each worker copy)
# ══════════════════════════════════════════════════════════════════════════════════════════════════════
VLLM_GPUS, TP_SIZE = WORKER_GPU, 1
os.environ["CUDA_VISIBLE_DEVICES"] = VLLM_GPUS    # MUST be set before torch/vllm init -> done here, above `import torch`
# FlashInfer JIT-compiles its sampler with nvcc+ninja; on WSL's older apt CUDA toolkit that build fails.
# Use vLLM's native PyTorch sampler instead (negligible perf cost). Must be set before vllm is imported (§7).
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
# GPU 0 usually drives the display (~2.6 GB) -> leave it headroom; GPU 1 is headless compute.
_DEF_UTIL = "0.82" if WORKER_GPU == "0" else "0.90"
GPU_MEM_UTIL = float(os.environ.get("GPU_MEM_UTIL", _DEF_UTIL))
print(f"[gpu] GPU {WORKER_GPU} -> CUDA_VISIBLE_DEVICES={VLLM_GPUS} util={GPU_MEM_UTIL}", flush=True)

# Anchor cwd to the data folder. Default is the Windows G: path (same as the HF notebook); override with
# MADS_DATA_DIR on Linux/WSL where the corpus lives elsewhere. Guarded so a missing path won't error here.
_DATA_DIR = Path(os.environ.get("MADS_DATA_DIR",
              "../data"))
if not _DATA_DIR.exists():                       # Linux/WSL fallback: the setup script copies data to ~/mads-data
    _alt = Path.home() / "mads-data"
    if _alt.exists(): _DATA_DIR = _alt
if _DATA_DIR.exists(): os.chdir(_DATA_DIR)
HERE             = Path.cwd()
POSTS_PARQUET    = HERE / "all_posts.parquet"
COMMENTS_PARQUET = HERE / "all_comments.parquet"
# all_posts is read in §3; all_comments is ONLY needed to (re)build the assembly, which we REUSE from
# thr_comments.parquet — so it is optional here (we don't copy the 2 GB file to Linux).
assert POSTS_PARQUET.exists(), f"all_posts.parquet not found under {HERE}"

# Device for the (checkpointed) embedding anchoring stage. vLLM owns the GPU later; anchoring runs first
# and frees itself before vLLM loads. If the anchoring checkpoints already exist this is never used.
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

# --- Semantic anchoring (Stage A) ---
EMBED_MODEL        = "all-MiniLM-L6-v2"
EMBED_SIM_THRESHOLD= 0.45
EMBED_ALL_POSTS    = True

# --- LLM (Stage C) --- pinned for reproducibility
LLM_MODEL          = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MODEL_TAG          = "7B-AWQ"   # REUSE the existing anchoring/assembly checkpoints in relevance_filtering-7B-AWQ
VTAG               = "-vllm-v5" # v5 namespace -> isolated from v1/v2/v3/v4. v5 SEEDS from v1's parsed=True labels
                                # and re-labels ONLY v1's parsed=False rows with v1's prompt in single-object form.
PRIOR_COMMENTS     = "labeled_comments-vllm.parquet"   # v1's assembled comment labels (id, about_topic, stance, parsed, link_id)
PRIOR_POSTS        = "labeled_posts-vllm.parquet"      # v1's assembled post labels
LLM_REVISION       = None       # pin to a commit hash for a frozen run
PROMPT_VERSION     = "v5-v1prompt-object"  # v1's zero-shot judgment text, guided single-JSON-OBJECT output (see §7b)
GUIDED_DECODING    = True       # constrain output to the JSON-OBJECT schema (valid JSON + valid enums; [] is illegal)
MAX_NEW_TOKENS     = int(os.environ.get("MAX_NEW_TOKENS", "2048"))  # JSON array over a whole chunk is far longer than terse
MAX_INPUT_TOKENS   = int(os.environ.get("MAX_INPUT_TOKENS", "6144")) # ~2.2k of cached few-shot examples + thread content
MAX_MODEL_LEN      = int(os.environ.get("MAX_MODEL_LEN", str(MAX_INPUT_TOKENS + MAX_NEW_TOKENS)))  # vLLM ctx (~8192)
# Cap concurrent sequences vLLM admits per step. vLLM's default (256) over-admits long sequences relative to
# the 4090 KV-cache budget, which triggers preemption->recompute THRASHING (throughput collapses to ~0 toks/s,
# repeated shm_broadcast 60s timeouts) on heavy thread groups. ~96 keeps the working set inside KV so big
# groups process in waves instead of thrashing. Raise toward 128-160 if light groups feel throttled.
MAX_NUM_SEQS       = int(os.environ.get("MAX_NUM_SEQS", "128"))
CHAR_BUDGET        = 3000   # v4: with MAX_TARGETS_PER_CHUNK=1 a chunk is one target + its context chain; this only
                           # bounds the context size. Qwen-7B + arrays OMIT ids in big chunks -> v4 uses 1 obj/call.
MAX_TARGETS_PER_CHUNK = int(os.environ.get("MAX_TARGETS_PER_CHUNK", "1"))  # hard cap on labelable ids per call (the
                           # omission driver). 1 = ONE comment per JSON array -> the model cannot drop an id (omission-proof).
REDO_OMITTED       = True   # §9c: after the main pass, re-label every comment left parsed=False (an omission)
MAX_BODY_CHARS     = 400
CTX_FULL_LEVELS    = 3
CONTEXT_BODY_CHARS = 80

# --- Scope: which threads to send to the LLM ---
# True  = all 85K *anchored* threads (lexical OR semantic hit) — recommended with vLLM; recovers ~62% of
#         the estimated true relevant population vs ~13% for tier-1 alone.
# False = tier-1 only (~11,642 threads) — the old HF speed compromise; still valid as a fast baseline.
CLASSIFY_ANCHORED  = True

# --- Caps (None = full corpus) ---
LIMIT_POSTS        = None
MAX_THREADS        = None

# --- Checkpointing / resumability ---
CKPT_DIR           = HERE / f"relevance_filtering-{MODEL_TAG}"   # holds the reused upstream anchoring/assembly artifacts
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR            = CKPT_DIR
SAVE_EVERY_THREADS = 250   # checkpoint every 250 threads -> a crash/reboot loses <=250 threads of work.
SAVE_EVERY_POSTS   = 600
BENCH_THREADS      = 500   # threads timed by the benchmark stage
CLAIM_SIZE         = SAVE_EVERY_THREADS   # DP worker (§9b): threads pulled per claim from the shared queue
RECLAIM_TIMEOUT_S  = 600   # DP worker: a dead worker's unfinished claims become re-claimable after this many seconds

print(f"[vllm-cfg] CUDA_VISIBLE_DEVICES={VLLM_GPUS} | gpu_mem_util={GPU_MEM_UTIL} "
      f"| max_model_len={MAX_MODEL_LEN} | data={HERE}", flush=True)

[gpu] GPU 1 -> CUDA_VISIBLE_DEVICES=1 util=0.9
[vllm-cfg] CUDA_VISIBLE_DEVICES=1 | gpu_mem_util=0.9 | max_model_len=8192 | data=~/mads-data
CPU times: user 2.33 s, sys: 618 ms, total: 2.95 s
Wall time: 2.36 s


### Checkpoint helpers

Crash-safe scheme: immutable `part_*.parquet` + atomically-rewritten JSON ledger. Every LLM-output name carries
the `VTAG` (`-vllm-v4`) suffix so it never collides with the v1/v2/v3 outputs.

In [2]:
%%time
def ck(name):       return CKPT_DIR / name
def have(name):     return ck(name).exists()
def save_parquet(df, name): df.to_parquet(ck(name))
def load_parquet(name):     return pd.read_parquet(ck(name))
def save_json(obj, name):   ck(name).write_text(json.dumps(obj))
def load_json(name):        return json.loads(ck(name).read_text()) if have(name) else None
def save_json_atomic(obj, name):
    tmp = ck(name + ".tmp"); tmp.write_text(json.dumps(obj)); tmp.replace(ck(name))

def append_parts(df, subdir):
    d = ck(subdir); d.mkdir(parents=True, exist_ok=True)
    k = len(list(d.glob("part_*.parquet")))
    df.to_parquet(d / f"part_{k:05d}.parquet")
def load_parts(subdir):
    d = ck(subdir)
    if not d.exists(): return pd.DataFrame()
    files = sorted(d.glob("part_*.parquet"))
    return pd.concat((pd.read_parquet(f) for f in files), ignore_index=True) if files else pd.DataFrame()
def load_parts_all(base):    # merge a stage across ALL part-dirs whose name starts with `base`
    dfs = [load_parts(d.name) for d in sorted(CKPT_DIR.glob(f"{base}*")) if d.is_dir()]
    dfs = [x for x in dfs if len(x)]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# vLLM-namespaced LLM outputs (VTAG="-vllm-v4") -> isolated from v1/v2/v3.
COMMENT_LABELS = f"comment_labels{VTAG}"
COMMENT_DONE   = f"comment_done_threads{VTAG}.json"
POST_LABELS    = f"post_labels{VTAG}"
POST_DONE      = f"post_done_ids{VTAG}.json"
LLM_CACHE      = f"llm_cache{VTAG}-dp{WORKER_GPU}.json" if WORKER_MODE else f"llm_cache{VTAG}.json"  # per-worker -> no clobber
REL_POSTS_OUT  = f"relevant_posts{VTAG}.parquet"
REL_COMM_OUT   = f"relevant_comments{VTAG}.parquet"
MANIFEST_OUT   = f"manifest{VTAG}.json"

def load_vllm_comments():
    """v5 comment labels — SUCCESSFULLY-LABELED ONLY. Globs the v5 namespace (seed = v1-good + main reprocess +
    `-dp0/-dp1` workers), dedups PREFERRING `parsed=True` (a real label OR a §E-routed label wins over a
    parse-failure for the same id), then **DROPS every remaining `parsed=False` row**. So this frame is v1's kept
    labels plus the reprocessed omissions; anything still unparsed is ABSENT. §13 reports the coverage + reasons."""
    df = load_parts_all(f"comment_labels{VTAG}")
    if "parsed" in df.columns:
        df = df.sort_values("parsed", kind="stable")          # False before True -> keep='last' keeps a True if any
    df = df.drop_duplicates("id", keep="last")
    if "parsed" in df.columns:
        df = df[df["parsed"].astype(bool)].reset_index(drop=True)   # DROP omissions/parse-failures from the final set
    return df

print("checkpoint dir:", CKPT_DIR, "| labels:", COMMENT_LABELS, "| ledger:", COMMENT_DONE, flush=True)

checkpoint dir: ~/mads-data/relevance_filtering-7B-AWQ | labels: comment_labels-vllm-v5 | ledger: comment_done_threads-vllm-v5.json
CPU times: user 1.5 ms, sys: 0 ns, total: 1.5 ms
Wall time: 1.08 ms


## 2 · Detection vocabulary

Tier-1 = unambiguous (auto-trusted anchors). Candidate = deliberately broad recall net. RE2-safe so the same
pattern runs in Polars and DuckDB.

In [3]:
%%time
HIGH_PRECISION_TERMS = [
    r"congestion pricing", r"congestion toll(?:ing)?", r"congestion fee", r"congestion charge",
    r"congestion relief zone", r"\bCRZ\b", r"\bcbdtp\b",
    r"central business district toll(?:ing)?", r"traffic mobility act",
    r"hochul['’]?s? (?:congestion )?toll", r"toll(?:ing)? (?:pause|freeze)",
    r"pause(?:d|s)? the (?:congestion )?toll",
    r"\$9 (?:toll|charge|fee|congestion|cordon)", r"\$15 (?:toll|charge|fee|congestion|cordon)",
    r"(?:nine|9)[ -]?dollar (?:congestion )?toll", r"(?:fifteen|15)[ -]?dollar (?:congestion )?toll",
]
HIGH_PRECISION_PATTERN = "(?:" + "|".join(HIGH_PRECISION_TERMS) + ")"

CANDIDATE_TERMS = [
    r"\btoll", r"congestion", r"\bMTA\b", r"hochul", r"60th st", r"\$9\b", r"\$15\b",
    r"gridlock", r"traffic mobility", r"central business district", r"cordon", r"gantr",
    r"manhattan.{0,15}(?:driv|commut|toll)",
]
CANDIDATE_PATTERN = "(?:" + "|".join(CANDIDATE_TERMS) + ")"
print("Tier-1 terms:", len(HIGH_PRECISION_TERMS), "| candidate terms:", len(CANDIDATE_TERMS))

Tier-1 terms: 16 | candidate terms: 13
CPU times: user 48 μs, sys: 84 μs, total: 132 μs
Wall time: 126 μs


## 3 · Load & clean posts

In [4]:
%%time
import polars as pl
posts = pl.read_parquet(POSTS_PARQUET, columns=[
    "id","name","subreddit","title","selftext","created_utc","score","num_comments","author","permalink"])
if LIMIT_POSTS: posts = posts.head(LIMIT_POSTS)
JUNK = ["[removed]","[deleted]","[deleted by user]","[ Removed by Reddit ]"]
posts = posts.with_columns(
    pl.when(pl.col("selftext").is_in(JUNK)).then(pl.lit("")).otherwise(pl.col("selftext")).fill_null("").alias("selftext_clean")
).with_columns(
    (pl.col("title").fill_null("") + pl.lit(" ") + pl.col("selftext_clean")).alias("full_text"))
print(f"Loaded {posts.height:,} posts")

Loaded 996,115 posts
CPU times: user 858 ms, sys: 1.28 s, total: 2.14 s
Wall time: 574 ms


## 4 · Stage A — anchor detection

A *thread* is **anchored** if the post or any comment is topical, by either the lexical net or semantic
similarity to the Tier-1 centroid. These stages are CHECKPOINTED — on this machine they load instantly
from the existing `relevance_filtering-7B-AWQ/` artifacts (no GPU recompute).

In [5]:
%%time
# A1 — lexical anchors on posts (cheap; always runs)
posts = posts.with_columns([
    pl.col("full_text").str.contains("(?i)"+HIGH_PRECISION_PATTERN).alias("hp"),
    pl.col("full_text").str.contains("(?i)"+CANDIDATE_PATTERN).alias("cand"),
])
lex_anchor_posts = posts.filter(pl.col("cand"))
tier1_posts      = posts.filter(pl.col("hp"))
print(f"[A1] lexical post anchors: {lex_anchor_posts.height:,}  (Tier-1: {tier1_posts.height:,})")

[A1] lexical post anchors: 20,311  (Tier-1: 3,589)
CPU times: user 1min 10s, sys: 309 ms, total: 1min 10s
Wall time: 5.94 s


In [6]:
%%time
# A2 — semantic anchors on posts (cosine sim to Tier-1 centroid). CHECKPOINTED + chunked.
if have("sem_anchors.parquet"):
    _sa = load_parquet("sem_anchors.parquet")
    sem_anchor_post_ids = set(_sa.loc[_sa["sem"] >= EMBED_SIM_THRESHOLD, "id"])
    post_sem = dict(zip(_sa["id"], _sa["sem"].astype(float)))
    print(f"[SKIP] loaded {len(_sa):,} post sem-scores from checkpoint; anchors={len(sem_anchor_post_ids):,}", flush=True)
else:
    import torch
    from sentence_transformers import SentenceTransformer, util
    print("[A2a] loading embedding model ...", flush=True)
    emb_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
    t1_texts = tier1_posts.select("full_text").to_series().to_list()
    print(f"[A2b] centroid from {len(t1_texts):,} Tier-1 posts ...", flush=True)
    t1_emb = emb_model.encode(t1_texts, batch_size=256, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False, device=DEVICE)
    centroid = t1_emb.mean(dim=0, keepdim=True); centroid = centroid / centroid.norm()
    scope = posts if EMBED_ALL_POSTS else lex_anchor_posts
    total_rows = scope.height
    print(f"[A2c] encoding {total_rows:,} posts on GPU in chunks ...", flush=True)
    DF_CHUNK_SIZE = 100_000
    ids_all, sem_all = [], []
    for offset in range(0, total_rows, DF_CHUNK_SIZE):
        pl_chunk    = scope.select(["id","full_text"]).slice(offset, DF_CHUNK_SIZE)
        chunk_texts = pl_chunk["full_text"].to_list()
        chunk_ids   = pl_chunk["id"].to_list()
        pe = emb_model.encode(chunk_texts, batch_size=256, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False, device=DEVICE)
        sem_scores = util.cos_sim(pe, centroid).squeeze(1).cpu().numpy()
        ids_all += chunk_ids; sem_all += sem_scores.astype(float).tolist()
        print(f"   [A2c] {min(offset+DF_CHUNK_SIZE, total_rows):,}/{total_rows:,}", flush=True)
    _sa = pd.DataFrame({"id": ids_all, "sem": sem_all})
    save_parquet(_sa, "sem_anchors.parquet")
    sem_anchor_post_ids = set(_sa.loc[_sa["sem"] >= EMBED_SIM_THRESHOLD, "id"])
    post_sem = dict(zip(_sa["id"], _sa["sem"].astype(float)))
    print(f"[A2] semantic post anchors (>= {EMBED_SIM_THRESHOLD}): {len(sem_anchor_post_ids):,}", flush=True)
    del emb_model, t1_emb, centroid
    try: del pe
    except NameError: pass
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("[A2] freed embedding model from GPU", flush=True)

[SKIP] loaded 996,115 post sem-scores from checkpoint; anchors=17,640
CPU times: user 635 ms, sys: 125 ms, total: 760 ms
Wall time: 747 ms


In [7]:
%%time
# A3 — lexical anchors on comments (DuckDB streams the 2 GB file). CHECKPOINTED.
if have("comment_anchors.parquet"):
    canc = load_parquet("comment_anchors.parquet")
    print(f"[SKIP] loaded {len(canc):,} comment anchors from checkpoint", flush=True)
else:
    import duckdb
    con = duckdb.connect()
    canc = con.execute(f"""
        SELECT id, link_id,
               regexp_matches(body, $hp,   'i') AS hp,
               regexp_matches(body, $cand, 'i') AS cand
        FROM read_parquet('{COMMENTS_PARQUET.as_posix()}')
        WHERE regexp_matches(body, $cand, 'i')
    """, {"hp": HIGH_PRECISION_PATTERN, "cand": CANDIDATE_PATTERN}).df()
    save_parquet(canc, "comment_anchors.parquet")
    print(f"[A3] lexical comment anchors: {len(canc):,}  (Tier-1: {int(canc['hp'].sum()):,})", flush=True)
anchor_comment_ids_by_thread = canc.groupby("link_id")["id"].apply(set).to_dict()

[SKIP] loaded 218,940 comment anchors from checkpoint
CPU times: user 1.29 s, sys: 40.1 ms, total: 1.33 s
Wall time: 1.33 s


## 5 · Capture–recapture recall estimate (no labels)

Two independent detectors at the **thread** level: lexical vs semantic. Their overlap estimates the hidden
total of topical threads (Lincoln–Petersen), hence the candidate net's recall.

In [8]:
%%time
def t3(i): return "t3_" + i
lex_threads = set(lex_anchor_posts.select("name").to_series().to_list()) | set(canc["link_id"].unique())
sem_threads = {t3(i) for i in sem_anchor_post_ids}
A, B = len(lex_threads), len(sem_threads); m = len(lex_threads & sem_threads)
union = len(lex_threads | sem_threads)
N_hat = (A * B / m) if m else None
est_recall = (union / N_hat) if N_hat else None
print(f"lexical n_A={A:,} | semantic n_B={B:,} | overlap m={m:,}")
print(f"N_hat ~ {N_hat:,.0f} | union={union:,} | est candidate-net recall ~ {est_recall:.1%}"
      if m else "overlap=0, cannot estimate")
save_json({"n_lexical": A, "n_semantic": B, "overlap": m, "union": union,
           "N_hat": (round(N_hat) if N_hat else None),
           "est_recall": (round(est_recall, 4) if est_recall else None)}, "capture_recapture.json")

if have("anchored_threads.json"):
    anchored_threads = set(load_json("anchored_threads.json"))
    print(f"[SKIP] loaded {len(anchored_threads):,} anchored threads from checkpoint")
else:
    anchored_threads = lex_threads | sem_threads
    if MAX_THREADS: anchored_threads = set(sorted(anchored_threads)[:MAX_THREADS])
    save_json(sorted(anchored_threads), "anchored_threads.json")
print(f"anchored threads to assemble: {len(anchored_threads):,}", flush=True)

if have("tier1_threads.json"):
    tier1_threads = set(load_json("tier1_threads.json"))
    print(f"[SKIP] loaded {len(tier1_threads):,} tier-1 threads from checkpoint")
else:
    t1_post_names = set(tier1_posts.select("name").to_series().to_list())
    t1_comment_threads = set(canc.loc[canc["hp"] == True, "link_id"].unique())
    tier1_threads = (t1_post_names | t1_comment_threads) & anchored_threads
    save_json(sorted(tier1_threads), "tier1_threads.json")
print(f"tier-1 threads for LLM classification: {len(tier1_threads):,}", flush=True)

# Derived: the actual set handed to the LLM (controlled by CLASSIFY_ANCHORED in §1).
# Set here (after both anchored_threads and tier1_threads exist) so all downstream cells use one variable.
classify_threads = anchored_threads if CLASSIFY_ANCHORED else tier1_threads
print(f"[scope] CLASSIFY_ANCHORED={CLASSIFY_ANCHORED} -> classify_threads={len(classify_threads):,} "
      f"({'all anchored' if CLASSIFY_ANCHORED else 'tier-1 only'})", flush=True)

lexical n_A=77,624 | semantic n_B=17,640 | overlap m=9,881
N_hat ~ 138,578 | union=85,383 | est candidate-net recall ~ 61.6%
[SKIP] loaded 85,383 anchored threads from checkpoint
anchored threads to assemble: 85,383
[SKIP] loaded 11,642 tier-1 threads from checkpoint
tier-1 threads for LLM classification: 11,642
[scope] CLASSIFY_ANCHORED=True -> classify_threads=85,383 (all anchored)
CPU times: user 47.1 ms, sys: 44.2 ms, total: 91.3 ms
Wall time: 84.9 ms


## 6 · Stage B — thread assembly

Pull every anchored thread's post + comments and group them. CHECKPOINTED to parquet (loads instantly here).

In [9]:
%%time
if have("thr_posts.parquet") and have("thr_comments.parquet"):
    thr_posts    = load_parquet("thr_posts.parquet")
    thr_comments = load_parquet("thr_comments.parquet")
    print(f"[SKIP] loaded assembly: {len(thr_posts):,} posts + {len(thr_comments):,} comments", flush=True)
else:
    import duckdb
    con = duckdb.connect()
    con.register("anchored", pd.DataFrame({"name": sorted(anchored_threads)}))
    thr_posts = con.execute(f"""
        SELECT p.id, p.name, p.title, p.selftext, p.subreddit, p.score, p.num_comments, p.hp_flag
        FROM (SELECT *, regexp_matches(coalesce(title,'')||' '||coalesce(selftext,''), $hp, 'i') AS hp_flag
              FROM read_parquet('{POSTS_PARQUET.as_posix()}')) p
        SEMI JOIN anchored a ON p.name = a.name
    """, {"hp": HIGH_PRECISION_PATTERN}).df()
    thr_comments = con.execute(f"""
        SELECT c.id, c.parent_id, c.link_id, c.body, c.author, c.created_utc, c.score
        FROM read_parquet('{COMMENTS_PARQUET.as_posix()}') c
        SEMI JOIN anchored a ON c.link_id = a.name
    """).df()
    save_parquet(thr_posts, "thr_posts.parquet")
    save_parquet(thr_comments, "thr_comments.parquet")
    print(f"[B] assembled {len(thr_posts):,} posts + {len(thr_comments):,} comments", flush=True)
comments_by_thread = {k: v for k, v in thr_comments.groupby("link_id")}
posts_by_name = thr_posts.set_index("name").to_dict("index")

[SKIP] loaded assembly: 84,854 posts + 5,778,767 comments
CPU times: user 9.19 s, sys: 5.65 s, total: 14.8 s
Wall time: 11.5 s


## 7 · Stage C — vLLM (lazy load)

Greedy decoding via vLLM. `ensure_llm()` builds one `LLM` engine (AWQ weights, fp16, Marlin kernels) on the
first call. vLLM does its own continuous batching, so there is **no manual batch size and no OOM-backoff** —
we just hand `llm.generate()` the whole group of prompts and it schedules them to fill the GPU. The on-disk
prompt cache (`llm_cache-vllm.json`) is flushed at every checkpoint.

In [10]:
%%time
from vllm import LLM, SamplingParams

# Guided-decoding schema: a JSON ARRAY of per-target label objects. Constrains the decoder so EVERY output is
# valid JSON with valid enums (stance in the 5 values, about_topic/refers_to_parent in {0,1}) -> eliminates the
# parse failures v1's terse format produced. The array length and id strings are NOT fixed by the schema, so the
# parser still maps ids back to inputs and defaults any the model omitted.
# v4: ONE target per call -> a single JSON OBJECT (no array). Because the schema is an object, an empty array `[]`
# is grammatically IMPOSSIBLE under guided decoding, so the model can no longer go silent on an off-topic target —
# off-topic must be expressed as about_topic=0. No `id` field: the caller knows the one target's id by position.
LABELS_SCHEMA = {
    "type": "object",
    "properties": {
        "about_topic": {"type": "integer", "enum": [0, 1]},
        "stance": {"type": "string", "enum": ["pro", "anti", "neutral", "mixed", "NA"]},
        "refers_to_parent": {"type": "integer", "enum": [0, 1]},
        "confidence": {"type": "number"}},
    "required": ["about_topic", "stance", "confidence"],
}
def _build_sp():
    base = dict(temperature=0.0, max_tokens=MAX_NEW_TOKENS)
    if not GUIDED_DECODING:
        return SamplingParams(**base)
    # vLLM's structured-output param moved across versions (V1 'structured_outputs'/StructuredOutputsParams;
    # older 'guided_decoding'/GuidedDecodingParams; class importable from vllm or vllm.sampling_params). Try the
    # known combos; if none take, fall back to free output (the JSON parser still validates every row).
    import importlib
    for mod, cls, field in (("vllm.sampling_params","StructuredOutputsParams","structured_outputs"),
                            ("vllm","StructuredOutputsParams","structured_outputs"),
                            ("vllm.sampling_params","GuidedDecodingParams","guided_decoding"),
                            ("vllm","GuidedDecodingParams","guided_decoding")):
        try:
            klass = getattr(importlib.import_module(mod), cls)
            sp = SamplingParams(**base, **{field: klass(json=LABELS_SCHEMA)})
            print(f"[vllm] guided decoding ON via {cls} (.{field})", flush=True)
            return sp
        except Exception:
            continue
    print("[vllm] guided decoding API not found -> free output; the JSON parser still validates every row", flush=True)
    return SamplingParams(**base)

_LLM = {"llm": None, "sp": None, "tok": None}
def ensure_llm():
    if _LLM["llm"] is None:
        print(f"[vllm] loading {LLM_MODEL} | TP={TP_SIZE} | gpu_mem_util={GPU_MEM_UTIL} "
              f"| max_model_len={MAX_MODEL_LEN} | CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}", flush=True)
        import shutil
        # enforce_eager skips torch.compile (needs the CUDA toolkit / nvcc, absent on default WSL).
        # Auto: eager when nvcc is missing; override with env VLLM_EAGER=0/1.
        _ev = os.environ.get("VLLM_EAGER")
        _eager = bool(int(_ev)) if _ev is not None else (shutil.which("nvcc") is None)
        print(f"[vllm] enforce_eager={_eager} (nvcc {'found' if shutil.which('nvcc') else 'absent'})", flush=True)
        kw = dict(model=LLM_MODEL, tokenizer=LLM_MODEL, quantization="awq_marlin", dtype="float16",
                  tensor_parallel_size=TP_SIZE, gpu_memory_utilization=GPU_MEM_UTIL,
                  max_model_len=MAX_MODEL_LEN, max_num_seqs=MAX_NUM_SEQS,   # cap concurrency -> no KV preempt thrash
                  trust_remote_code=True, seed=0, enforce_eager=_eager)
        if LLM_REVISION: kw["revision"] = LLM_REVISION
        _LLM["llm"] = LLM(**kw)
        _LLM["tok"] = _LLM["llm"].get_tokenizer()
        _LLM["sp"]  = _build_sp()       # guided JSON-array decoding (or plain if unavailable)
    return _LLM["llm"], _LLM["sp"], _LLM["tok"]

_cache = load_json(LLM_CACHE) or {}
def _key(prompt): return hashlib.sha1(f"{LLM_MODEL}|{PROMPT_VERSION}|{prompt}".encode()).hexdigest()
def flush_cache(): save_json_atomic(_cache, LLM_CACHE)

def _split_prompt(p, parts=2):
    """Split an over-long rendered prompt into `parts` sub-prompts that EACH repeat the instructions + post
    header and divide the comment lines between them — so every comment is still labeled (vs. truncation,
    which drops the trailing ones). Used only on the rare length-error path. Splits on the 'COMMENTS:' marker
    so the header rides along with each piece; falls back to a plain line split if the marker is absent."""
    marker = "COMMENTS:\n"
    k = p.find(marker)
    head, body = (p[:k+len(marker)], p[k+len(marker):]) if k != -1 else ("", p)
    lines = [ln for ln in body.split("\n") if ln != ""]
    if len(lines) < 2: return [p]                    # single line: indivisible (caller truncates as last resort)
    sz = math.ceil(len(lines) / parts)
    return [head + "\n".join(lines[i:i+sz]) for i in range(0, len(lines), sz)]

def llm_generate(prompts, outer=0, inner=0):
    """Cache-aware batched generation. Uncached prompts are chat-templated and handed to vLLM in ONE
    generate() call; vLLM batches them internally (continuous batching). Outputs come back in input order."""
    llm, sp, tok = ensure_llm()
    out = [None]*len(prompts); todo, idx = [], []
    for i, p in enumerate(prompts):
        k = _key(p)
        if k in _cache: out[i] = _cache[k]
        else: todo.append(p); idx.append(i)
    if todo:
        chats = [tok.apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
                 for p in todo]
        try:
            texts = [r.outputs[0].text for r in llm.generate(chats, sp)]   # COMMON PATH: no pre-tokenize
        except Exception:
            # RARE PATH ONLY. One over-long prompt makes vLLM raise VLLMValidationError and abort the WHOLE
            # batch — which previously killed the run and left the GPU idle for hours. CHAR_BUDGET bounds chunks
            # by characters, but token-dense text (CJK, emoji, code) can blow past the TOKEN budget. So only now
            # do we pay to tokenize, and instead of truncating we SPLIT each offender into pieces that fit, run
            # them, and re-join their outputs per original prompt (parse_item then re-parses the object). Self-healing:
            # the run continues. If nothing is actually over-long, it's a different error — re-raise, don't mask.
            _budget = MAX_INPUT_TOKENS - 128
            def _fits(s): return len(tok(s, add_special_tokens=False).get("input_ids", [])) <= _budget
            pieces, owners, nover = [], [], 0
            for t, p in enumerate(todo):
                if _fits(p):
                    pieces.append(p); owners.append(t); continue
                nover += 1
                work = _split_prompt(p, 2)
                while work:
                    s = work.pop(0)
                    if _fits(s):
                        pieces.append(s); owners.append(t); continue
                    sub = _split_prompt(s, 2)
                    if len(sub) <= 1:                # _split_prompt couldn't divide it (single comment line):
                        ids = tok(s, add_special_tokens=False).get("input_ids", [])  # indivisible -> truncate
                        pieces.append(tok.decode(ids[:_budget])); owners.append(t)   # (true last resort)
                    else:
                        work = sub + work            # divisible -> recurse on the smaller pieces
            if nover == 0:
                raise                                # not a length problem — surface the real error
            print(f"[llm] {nover} over-long prompt(s) split into fitting pieces; retrying batch", flush=True)
            pchats = [tok.apply_chat_template([{"role":"user","content":s}], tokenize=False, add_generation_prompt=True)
                      for s in pieces]
            pres = [r.outputs[0].text for r in llm.generate(pchats, sp)]
            merged = {}
            for own, txt in zip(owners, pres): merged.setdefault(own, []).append(txt)
            def _merge_obj(parts):    # v4: ONE target per prompt -> pick the first piece that yields a usable object
                for p in parts:
                    lab, _r = parse_item(p)
                    if lab: return p
                return parts[0] if parts else "{}"
            texts = [_merge_obj(merged[t]) for t in range(len(todo))]    # one JSON object per (single-target) prompt
        for j, txt in zip(idx, texts):
            out[j] = txt; _cache[_key(prompts[j])] = txt
    return out

_STANCE_SET  = {"pro","anti","neutral","mixed","NA"}
_STANCE_NORM = {"pro":"pro","anti":"anti","neutral":"neutral","mixed":"mixed","na":"NA"}
def _b01(s):
    if isinstance(s, bool): return s
    if isinstance(s, (int, float)): return int(s) == 1
    return str(s).strip().lower() in ("1","true","yes","y","t")
def _conf(s):
    try: return max(0.0, min(1.0, float(s)))
    except Exception: return None
def parse_item(text: str):
    """Parse the guided single-OBJECT output for ONE target:
        {"about_topic","stance","refers_to_parent","confidence"}
    Returns (label, fail_reason). On success label={about_topic,stance,confidence,refers_to_parent} and
    fail_reason="". On failure label={} and fail_reason is one of:
       "empty"          -> blank output or an empty array/object (model declined to emit a verdict)
       "malformed"      -> no parseable JSON object in the text
       "missing_fields" -> parsed an object but it lacks the required about_topic
    Splitting the failures (G) lets the bench/§13 show WHY a target is unlabeled instead of one opaque count.
    Stays defensive: unwraps a stray [ {...} ] array, salvages an object embedded in prose, normalizes stance."""
    s = (text or "").strip()
    if not s or s in ("[]", "{}", "[ ]", "{ }"):
        return {}, "empty"
    try:
        v = json.loads(s)
    except Exception:
        a, b = s.find("{"), s.rfind("}")                 # salvage an object embedded in stray prose
        if a != -1 and b > a:
            try: v = json.loads(s[a:b+1])
            except Exception: return {}, "malformed"
        else:
            return {}, "malformed"
    if isinstance(v, list):                              # model still wrapped it in an array -> unwrap first elem
        if not v: return {}, "empty"
        v = v[0]
    if not isinstance(v, dict):
        return {}, "malformed"
    if "about_topic" not in v:
        return {}, "missing_fields"
    st = str(v.get("stance", "NA")).strip()
    st = _STANCE_NORM.get(st.lower(), st if st in _STANCE_SET else "NA")
    label = {"about_topic": _b01(v.get("about_topic", 0)), "stance": st,
             "confidence": _conf(v.get("confidence", 0.0)) or 0.0,
             "refers_to_parent": _b01(v.get("refers_to_parent", 0))}
    return label, ""

CPU times: user 4.15 s, sys: 2.09 s, total: 6.23 s
Wall time: 5.62 s


## 7b · Prompt — v1's judgment, guided single JSON object

**Verbatim v1 judgment, new format.** The instruction text below is copied from the v1 notebook (the relevance +
stance judgment that matched the hand-labeled gold best) — zero-shot, no few-shot examples, so we reproduce v1's
calls rather than changing them. The ONLY change is the OUTPUT: instead of v1's terse 5-field free-text line (which
failed to parse ~18% of the time), each call returns a guided single JSON object, so a parse failure is impossible.

In [ ]:
%%time
TOPIC = ("New York City congestion pricing: the CBDTP / the ~$9 toll to drive into the Manhattan "
         "'Congestion Relief Zone' below 60th St, the MTA program, and Hochul's pause/revival of it.")

# v1's COMMENT judgment text (verbatim), with v1's terse OUTPUT-FORMAT paragraph swapped for a single-object spec.
COMMENT_INSTRUCTIONS = (
 f"You label Reddit comments about {TOPIC}\n"
 "Use the thread structure to resolve what each comment refers to. A comment IS about the topic if, read in "
 "context, it engages with congestion pricing - INCLUDING short replies like 'me too', 'exactly', 'this' when "
 "they agree/disagree with a parent that is about it. A comment is NOT about the topic if it is an off-topic "
 "remark even inside a relevant thread (e.g. 'happy cake day', unrelated tangents).\n"
 "Lines shown as '(ctx, author): ...' are ancestor comments included ONLY for context - do NOT label "
 "them; only judge the single comment shown as [cID].\n"
 "Decide: about_topic (1 or 0); stance in {pro, anti, neutral, mixed, NA} "
 "(pro=supports the toll, anti=opposes, neutral=on-topic but no position e.g. a question, mixed=both sides, "
 "NA=not about_topic). Resolve agreement/disagreement RELATIVE TO THE PARENT it replies to "
 "(e.g. 'me too' under an anti comment is anti). refers_to_parent=1 if you needed the parent/context to decide else 0. "
 "confidence in 0..1. If a referent is '[deleted]'/'[removed]' and unresolvable, set about_topic=0, "
 "stance=NA, low confidence.\n"
 "OUTPUT: return ONLY a single JSON object for the one marked [cID], exactly "
 
 '{"about_topic": 0 or 1, "stance": "pro|anti|neutral|mixed|NA", "refers_to_parent": 0 or 1, "confidence": 0.0-1.0}. '
 "Do NOT wrap it in an array; do NOT add an id; label none of the (ctx) lines; return nothing but the object.")

# v1's POST judgment text (verbatim), terse OUTPUT-FORMAT swapped for a single-object spec.
POST_INSTRUCTIONS = (
 f"You label whether each Reddit POST is about {TOPIC}\n"
 "Decide: about_topic (1 or 0), stance in {pro, anti, neutral, mixed, NA} "
 "(pro=supports the toll, anti=opposes, neutral=on-topic but no position, mixed=both sides, NA=not about_topic), "
 "confidence in 0..1.\n"
 "OUTPUT: return ONLY a single JSON object for the one marked [pID], exactly "
 '{"about_topic": 0 or 1, "stance": "pro|anti|neutral|mixed|NA", "refers_to_parent": 0, "confidence": 0.0-1.0}. '
 "Do NOT wrap it in an array; do NOT add an id; return nothing but the object.")
print("prompts ready, version", PROMPT_VERSION, "| guided=", GUIDED_DECODING, "| chars:",
      len(COMMENT_INSTRUCTIONS), "/", len(POST_INSTRUCTIONS))

prompts ready, version v5-v1prompt-object | guided= True | chars: 1570 / 686
CPU times: user 145 μs, sys: 0 ns, total: 145 μs
Wall time: 139 μs


## 8 · Thread rendering (lineage-bounded chunks)

Identical chunking to the HF notebook: each comment is labeled in its **lineage context** (root→parent shown
as read-only `(ctx, …)` lines), chunks bounded per comment by `CHAR_BUDGET`, ancestor chain re-seeded across
chunk splits so no comment loses its parent.

In [12]:
%%time
def _clean(b):
    b = (b or "")
    return b if b not in JUNK else "[removed]"

def build_forest(cdf):
    nodes = {r.id: {"id": r.id, "parent_id": r.parent_id, "author": r.author,
                    "body": _clean(r.body), "created_utc": r.created_utc, "children": []}
             for r in cdf.itertuples()}
    roots = []
    for n in nodes.values():
        pid = n["parent_id"]
        if isinstance(pid, str) and pid.startswith("t1_") and pid[3:] in nodes:
            nodes[pid[3:]]["children"].append(n)
        else:
            roots.append(n)
    def srt(ns):
        ns.sort(key=lambda x: x["created_utc"])
        for x in ns: srt(x["children"])
    srt(roots)
    return nodes, roots

def _target_ids(nodes, roots, post_is_anchor, anchor_cids):
    if post_is_anchor or not anchor_cids:
        return set(nodes)
    keep = set()
    for a in anchor_cids:
        if a not in nodes: continue
        cur = a
        while cur in nodes:
            keep.add(cur); p = nodes[cur]["parent_id"]
            cur = p[3:] if isinstance(p, str) and p.startswith("t1_") and p[3:] in nodes else None
        stack = [nodes[a]]
        while stack:
            x = stack.pop(); keep.add(x["id"]); stack += x["children"]
    return keep

def _parent_map(nodes):
    pm = {}
    for n in nodes.values():
        pid = n["parent_id"]
        pm[n["id"]] = pid[3:] if isinstance(pid, str) and pid.startswith("t1_") and pid[3:] in nodes else None
    return pm

def _depths(nodes, pm):
    d = {}
    def dep(i):
        if i in d: return d[i]
        p = pm[i]; d[i] = 0 if p is None else dep(p)+1; return d[i]
    for i in nodes: dep(i)
    return d

# ── §E. Deterministic pre-routing of no-signal targets ──────────────────────────────────────────────────────
# Some targets carry no labelable signal: deleted/removed bodies and "symbolic-only" bodies (emoji-only,
# URL-only, pure punctuation). The model is forced to emit something for these and tends to produce an ambiguous
# empty / low-confidence object, so instead we label them deterministically (about_topic=0 / stance=NA) WITHOUT a
# model call, and mark them routed=True. This shrinks the genuinely-ambiguous set and saves GPU. Conservative by
# design: anything with a single alphanumeric character in ANY language still goes to the model (no language or
# length heuristics), so on-topic short replies and non-English comments are NEVER auto-routed.
_ROUTED_LABEL = {"about_topic": False, "stance": "NA", "refers_to_parent": False, "confidence": 1.0}
def _route_degenerate(body):
    """Return _ROUTED_LABEL if `body` has no labelable content, else None (-> send to the model)."""
    s = (body or "").strip()
    if s in ("", "[removed]", "[deleted]"):              # deleted/removed/empty (post-_clean these collapse here)
        return _ROUTED_LABEL
    t = re.sub(r"https?://\S+", "", s).strip()           # drop URLs; if nothing remains it was URL-only
    if t == "":
        return _ROUTED_LABEL
    if not any(ch.isalnum() for ch in t):                # emoji-only / pure punctuation / symbols (any script)
        return _ROUTED_LABEL
    return None

def render_chunks(post, cdf, anchor_cids, exact_targets=None):
    """Returns (chunks, routed). chunks = [(prompt_text, [single_id]), ...] for model labeling; routed =
    [(id, label_dict), ...] for the §E deterministic no-signal targets (never sent to the model).
    v5 reprocess: `exact_targets` (the v1 parse-failed comment ids for this thread) labels PRECISELY those ids;
    their ancestors are still shown as (ctx, ...) for the chain, but v1's already-good comments are not relabeled."""
    nodes, roots = build_forest(cdf)
    target = (set(exact_targets) & set(nodes)) if exact_targets is not None \
             else _target_ids(nodes, roots, post.get("post_is_anchor", False), anchor_cids)
    pm = _parent_map(nodes); depth = _depths(nodes, pm)
    header = f"POST [{post['id']}] {(post.get('title') or '')[:200]}\n{_clean(post.get('selftext'))[:800]}\n\nCOMMENTS:\n"

    def ind(i): return "  " * min(depth[i], 10)
    def ancestors(i):
        ch = []; cur = pm[i]
        while cur is not None: ch.append(cur); cur = pm[cur]
        return ch[::-1]
    def tgt_line(i):
        n = nodes[i]; return f"{ind(i)}[{i}] ({n['author']}): {n['body'][:MAX_BODY_CHARS]}"
    def ctx_line(i, dist):
        n = nodes[i]; cap = MAX_BODY_CHARS if dist <= CTX_FULL_LEVELS else CONTEXT_BODY_CHARS
        return f"{ind(i)}(ctx, {n['author']}): {n['body'][:cap]}"

    order = []
    for r in roots:
        st = [r]
        while st:
            x = st.pop()
            if x["id"] in target: order.append(x["id"])
            st.extend(reversed(x["children"]))
    # §E: split targets into deterministically-routed (no-signal) and real (model-labeled).
    routed = []
    real_order = []
    for i in order:
        lab = _route_degenerate(nodes[i]["body"])
        if lab is not None: routed.append((i, lab))
        else: real_order.append(i)
    order = real_order
    if not order: return [], routed

    hdr_len = len(header)
    chunks, lines, ids, shown, size = [], [], [], set(), hdr_len
    def flush():
        if ids: chunks.append((header + "\n".join(lines), list(ids)))
    for t in order:
        chain = ancestors(t)
        addl = [(a, ctx_line(a, depth[t]-depth[a])) for a in chain if a not in shown]
        tl   = tgt_line(t)
        cost = sum(len(l)+1 for _, l in addl) + len(tl) + 1
        if (size + cost > CHAR_BUDGET or len(ids) >= MAX_TARGETS_PER_CHUNK) and ids:   # cap targets/chunk -> no omissions
            flush(); lines, ids, shown, size = [], [], set(), hdr_len
            addl = [(a, ctx_line(a, depth[t]-depth[a])) for a in chain]
        for a, l in addl:
            lines.append(l); shown.add(a); size += len(l)+1
        lines.append(tl); shown.add(t); ids.append(t); size += len(tl)+1
    flush()
    return chunks, routed

def thread_to_chunks(name, only_cids=None):
    """Returns (chunks, idlists, routed): model prompts + their per-prompt id lists (one id each at
    MAX_TARGETS_PER_CHUNK=1), plus §E deterministically-routed (id, label) pairs for no-signal targets.
    v5 reprocess: `only_cids` = v1's parse-failed comment ids in this thread; only those are (re)labeled."""
    post = posts_by_name.get(name, {"id": name.replace("t3_",""), "title": "", "selftext": "", "hp_flag": False})
    post = dict(post)
    cdf = comments_by_thread.get(name)
    if cdf is None or len(cdf) == 0: return [], [], []
    post["post_is_anchor"] = bool(post.get("hp_flag")) or (name in lex_threads) or (name in sem_threads)
    anchor_cids = anchor_comment_ids_by_thread.get(name, set())
    chunks, idlists, routed = [], [], []
    _ch, routed = render_chunks(post, cdf, anchor_cids, exact_targets=only_cids)
    for text, ids in _ch:
        chunks.append(COMMENT_INSTRUCTIONS + "\n\n" + text); idlists.append(ids)
    return chunks, idlists, routed
print("rendering ready")

rendering ready
CPU times: user 72 μs, sys: 0 ns, total: 72 μs
Wall time: 67.5 μs


## 8b · Resumable classification functions

Same resumable ledger/part-file scheme as the HF notebook, but each checkpoint group's chunks are handed to
vLLM in **one** `llm_generate()` call (vLLM batches internally). Both functions resume from disk.

In [13]:
%%time
_REPROCESS_CIDS = None   # {thread: [v1 parse-failed comment ids]} set in §8d. classify renders ONLY those ids
                      # per thread (their ancestors shown as ctx), never the whole thread. None -> whole thread.
def classify_comment_threads(thread_names, save_every=SAVE_EVERY_THREADS, bench_limit=None,
                             labels_name=None, done_name=None, remaining_fn=None, cum=None):
    labels_name = labels_name or COMMENT_LABELS     # override to write to a separate namespace (the DP workers)
    done_name   = done_name   or COMMENT_DONE
    done = set(load_json(done_name) or [])
    todo = [n for n in thread_names if n not in done]
    if bench_limit is not None: todo = todo[:bench_limit]
    print(f"comment threads: {len(done):,} done | {len(todo):,} to process now", flush=True)
    if not todo: return
    ensure_llm()
    t0 = time.time(); start_done = len(done); prev_el = 0.0
    # `cum` (DP worker) is a caller-owned dict that persists ACROSS the many per-claim calls, so the cumulative
    # rate spans the whole worker session instead of resetting each claim. Single-engine §9 passes cum=None.
    if cum is not None:
        cum.setdefault("t0", t0); cum.setdefault("done0", start_done)
    for gs in range(0, len(todo), save_every):
        group = todo[gs:gs+save_every]
        chunks, idlists, routed = [], [], []
        for name in group:
            c, il, rt = thread_to_chunks(name, only_cids=(_REPROCESS_CIDS.get(name) if _REPROCESS_CIDS else None))
            chunks += c; idlists += il; routed += rt
        outs = llm_generate(chunks, outer=gs, inner=0) if chunks else []
        rows = []
        # §E routed targets: deterministic labels, parsed=True/routed=True, no model call.
        for cid, lab in routed:
            rows.append({"id": cid, "about_topic": bool(lab["about_topic"]), "stance": lab["stance"],
                "refers_to_parent": bool(lab["refers_to_parent"]), "confidence": float(lab["confidence"]),
                "parsed": True, "routed": True, "fail_reason": ""})
        # A model call = ONE target = ONE object (MAX_TARGETS_PER_CHUNK=1). parse_item gives (label, fail_reason).
        for out, ids in zip(outs, idlists):
            it, reason = parse_item(out)
            for k, cid in enumerate(ids):
                _it, _r = (it, reason) if k == 0 else ({}, "multi_target")   # cap=1 -> k>0 never happens
                rows.append({"id": cid,
                    "about_topic": bool(_it.get("about_topic", False)),
                    "stance": _it.get("stance", "NA"),
                    "refers_to_parent": bool(_it.get("refers_to_parent", False)),
                    "confidence": float(_it.get("confidence", 0.0) or 0.0),
                    "parsed": bool(_it), "routed": False, "fail_reason": _r})
        if rows: append_parts(pd.DataFrame(rows), labels_name)
        done |= set(group); save_json_atomic(sorted(done), done_name); flush_cache()
        now = time.time(); el = now - t0
        batch_rate = (el - prev_el) / max(len(group), 1); prev_el = el   # this group's own threads/sec
        if cum is not None:                                             # session-cumulative (across all claims)
            cum_el = now - cum["t0"]; cum_n = len(done) - cum["done0"]
        else:                                                           # single-call cumulative (§9 path)
            cum_el = el; cum_n = len(done) - start_done
        cum_rate = cum_el / max(cum_n, 1)
        # `remaining_fn` (DP worker) reports the TRUE shared-queue remaining across BOTH workers, recomputed from
        # the just-written ledgers; without it (single-engine §9) fall back to this run's batch-local count.
        _left = (f"{remaining_fn():,} left (shared queue)" if remaining_fn is not None
                 else f"{len(todo)-(gs+len(group)):,} left this run")
        print(f"[ckpt] +{len(group)} -> {len(done):,} threads | {cum_el:.0f}s | {cum_rate:.2f}s/thread cum "
              f"| {batch_rate:.2f}s/thread batch | {_left}", flush=True)

def classify_posts(ap_df, save_every=SAVE_EVERY_POSTS):
    done = set(load_json(POST_DONE) or [])
    todo = ap_df[~ap_df["id"].isin(done)].reset_index(drop=True)
    print(f"posts: {len(done):,} done | {len(todo):,} to process now", flush=True)
    if len(todo) == 0: return
    ensure_llm()
    t0 = time.time()
    for gs in range(0, len(todo), save_every):
        grp = todo.iloc[gs:gs+save_every]
        # A: ONE post per call -> one JSON object (no array, no omissions). §E routes no-signal posts (rare:
        # a post with no title AND no body) without a model call.
        prompts, pids, routed = [], [], {}
        for r in grp.itertuples():
            lab = _route_degenerate(f"{str(r.title)}  {str(r.selftext_clean)}")
            if lab is not None: routed[r.id] = lab; continue
            body = f"[{r.id}] {str(r.title)[:200]} :: {str(r.selftext_clean)[:500]}"
            prompts.append(POST_INSTRUCTIONS + "\n\nPOSTS:\n" + body); pids.append(r.id)
        outs = llm_generate(prompts, outer=gs, inner=0) if prompts else []
        parsed = {pid: parse_item(out) for pid, out in zip(pids, outs)}     # pid -> (label, fail_reason)
        rows = []
        for r in grp.itertuples():
            if r.id in routed:
                lab = routed[r.id]
                rows.append({"id": r.id, "about_topic": bool(lab["about_topic"]), "stance": lab["stance"],
                    "confidence": float(lab["confidence"]), "parsed": True, "routed": True, "fail_reason": ""})
            else:
                it, reason = parsed.get(r.id, ({}, "empty"))
                rows.append({"id": r.id,
                    "about_topic": bool(it.get("about_topic", bool(r.hp))),
                    "stance": it.get("stance", "NA"),
                    "confidence": float(it.get("confidence", 0.0) or 0.0),
                    "parsed": bool(it), "routed": False, "fail_reason": reason})
        append_parts(pd.DataFrame(rows), POST_LABELS)
        done |= set(grp["id"]); save_json_atomic(sorted(done), POST_DONE); flush_cache()
        print(f"[ckpt] posts {len(done):,} done | {time.time()-t0:.0f}s", flush=True)
print("classification functions ready")

classification functions ready
CPU times: user 74 μs, sys: 0 ns, total: 74 μs
Wall time: 68.9 μs


## 8d · Scope — seed v1, reprocess v1 parse-failures

Loads v1's assembled labels (`labeled_comments-vllm.parquet`), **copies every `parsed=True` row into the v5
namespace** (the seed — kept bit-for-bit), and builds the reprocess target: every `parsed=False` v1 row (a v1
omission), grouped by thread. Sets **`REPROCESS_THREADS`** (used by §8c/§9/§9b instead of `classify_threads`) and
**`_REPROCESS_CIDS`** so those stages re-label **only the omitted comment ids** with v1's prompt in single-object
form. Posts mirror this. Final v5 = v1-good ⊕ reprocessed omissions.

In [14]:
%%time
def _load_prior(path):
    df = load_parquet(path) if have(path) else pd.DataFrame()
    assert len(df), f"{path} not found / empty — v5 seeds from v1; run/locate the v1 output first"
    return df

# COMMENTS: seed v1 parsed=True; reprocess v1 parsed=False (by thread).
_v1c = _load_prior(PRIOR_COMMENTS)
_keep_cols = [c for c in ["id","about_topic","stance","refers_to_parent","confidence","parsed"] if c in _v1c.columns]
_good_c = _v1c[_v1c["parsed"].astype(bool)]
_bad_c  = _v1c[~_v1c["parsed"].astype(bool)]
print(f"[v5] v1 comments: {len(_v1c):,} | seed (parsed=True) {len(_good_c):,} "
      f"| reprocess (parsed=False) {len(_bad_c):,} ({len(_bad_c)/max(len(_v1c),1):.1%})", flush=True)
if not WORKER_MODE:                         # main notebook owns the one-time seed copy; workers must not race it
    _seeddir = ck(f"comment_labels{VTAG}-seed")
    if not _seeddir.exists():
        _seed = _good_c[_keep_cols].copy()
        if "routed" not in _seed.columns: _seed["routed"] = False
        if "fail_reason" not in _seed.columns: _seed["fail_reason"] = ""
        append_parts(_seed.reset_index(drop=True), f"comment_labels{VTAG}-seed")
        print(f"[v5] seeded {len(_seed):,} v1-good comment labels -> comment_labels{VTAG}-seed", flush=True)
    else:
        print(f"[v5] comment seed already present ({len(load_parts(f'comment_labels{VTAG}-seed')):,} rows) — skip", flush=True)
# reprocess targets: bad v1 comment ids -> their thread (link_id from the assembly)
_bad_ids = set(_bad_c["id"])
_bt = thr_comments[thr_comments["id"].isin(_bad_ids)][["id","link_id"]]
_REPROCESS_CIDS = _bt.groupby("link_id")["id"].apply(list).to_dict()
REPROCESS_THREADS = set(_REPROCESS_CIDS.keys())
_missing = len(_bad_ids) - int(_bt.shape[0])
print(f"[v5] reprocessing {len(_bad_ids):,} v1-omitted comments across {len(REPROCESS_THREADS):,} threads"
      + (f" ({_missing:,} ids not in assembly — skipped)" if _missing else ""), flush=True)

# POSTS: seed v1 parsed=True; reprocess v1 parsed=False.
_v1p = _load_prior(PRIOR_POSTS)
_pcols = [c for c in ["id","about_topic","stance","confidence","parsed"] if c in _v1p.columns]
_good_p = _v1p[_v1p["parsed"].astype(bool)] if "parsed" in _v1p.columns else _v1p
_bad_p  = _v1p[~_v1p["parsed"].astype(bool)] if "parsed" in _v1p.columns else _v1p.iloc[0:0]
if not WORKER_MODE:
    _pseed = ck(f"post_labels{VTAG}-seed")
    if not _pseed.exists():
        _ps = _good_p[_pcols].copy()
        if "routed" not in _ps.columns: _ps["routed"] = False
        if "fail_reason" not in _ps.columns: _ps["fail_reason"] = ""
        append_parts(_ps.reset_index(drop=True), f"post_labels{VTAG}-seed")
        print(f"[v5] seeded {len(_ps):,} v1-good post labels -> post_labels{VTAG}-seed", flush=True)
REPROCESS_POST_IDS = set(_bad_p["id"])
print(f"[v5] v1 posts: {len(_v1p):,} | seed {len(_good_p):,} | reprocess {len(REPROCESS_POST_IDS):,}", flush=True)

[v5] v1 comments: 5,778,767 | seed (parsed=True) 4,742,647 | reprocess (parsed=False) 1,036,120 (17.9%)
[v5] seeded 4,742,647 v1-good comment labels -> comment_labels-vllm-v5-seed
[v5] reprocessing 1,036,120 v1-omitted comments across 49,504 threads
[v5] seeded 31,348 v1-good post labels -> post_labels-vllm-v5-seed
[v5] v1 posts: 31,348 | seed 31,348 | reprocess 0
CPU times: user 6.81 s, sys: 2.09 s, total: 8.9 s
Wall time: 7.88 s


## 9b · Dynamic data-parallel worker (only runs when `WORKER_MODE`)

**Normal mode (`WORKER_MODE = False`): this cell does nothing** — execution falls through to the benchmark and the
single-engine §9 below.

**Worker mode (`WORKER_MODE = True`): this cell IS the run.** It pulls `CLAIM_SIZE` threads at a time from a shared
claim file (`dp_claims.json`, guarded by an `fcntl` lock), labels them into this worker's own namespace
(`comment_labels-vllm-dp{WORKER_GPU}`), releases them, and repeats until the queue is empty — then stops (so the
benchmark/§9/§11/§14 below are skipped in worker copies; do the merge from the **main** notebook).

Run two copies (GPU 0 and GPU 1); they share the one queue, so neither sits idle when the other finishes early.
**Interrupt** to pause a worker — it releases its in-flight claim and the engine stays loaded, so re-running this
cell resumes instantly. A worker killed outright has its unfinished claim reclaimed after `RECLAIM_TIMEOUT_S`.

In [15]:
%%time
# Claim-queue helpers (defined always; the loop at the bottom runs only in WORKER_MODE).
import contextlib
DP_LABELS = f"comment_labels{VTAG}-dp{WORKER_GPU}"          # this worker's private label part-dir
DP_DONE   = f"comment_done_threads{VTAG}-dp{WORKER_GPU}.json"
DP_CLAIMS = "dp_claims.json"                                # shared across workers
WID       = f"gpu{WORKER_GPU}"

@contextlib.contextmanager
def _claim_lock():
    # fcntl.flock is Linux-only and auto-releases if the process dies. Held ONLY for the ms-long claim/release
    # bookkeeping, never during generation. On Windows (smoke) fcntl is absent -> no-op (single-process is safe).
    try:
        import fcntl
        f = open(ck("dp_claims.lock"), "w")
        try: fcntl.flock(f, fcntl.LOCK_EX); yield
        finally: fcntl.flock(f, fcntl.LOCK_UN); f.close()
    except ImportError:
        yield

def _dp_all_done():
    # union of every v4 done-ledger (this worker + the other worker + the main notebook).
    s = set()
    for fp in CKPT_DIR.glob(f"comment_done_threads{VTAG}*.json"):
        try: s |= set(json.loads(fp.read_text()) or [])
        except Exception: pass
    return s

_DP_PENDING = None
def _dp_pending():
    # REPROCESS_THREADS is FROZEN for the run, so compute the pending order ONCE and cache it.
    global _DP_PENDING
    if _DP_PENDING is None:
        _DP_PENDING = sorted(REPROCESS_THREADS)
    return _DP_PENDING

def _claim_next(k):
    with _claim_lock():
        claims = load_json(DP_CLAIMS) or {}
        now = time.time(); done = _dp_all_done()
        active = {t for t, (w, ts) in claims.items() if now - ts < RECLAIM_TIMEOUT_S}
        batch = []
        for t in _dp_pending():
            if t in done or t in active: continue
            batch.append(t)
            if len(batch) >= k: break
        for t in batch: claims[t] = [WID, now]
        if batch: save_json_atomic(claims, DP_CLAIMS)
        return batch

def _release(batch):
    if not batch: return
    with _claim_lock():
        claims = load_json(DP_CLAIMS) or {}
        for t in batch: claims.pop(t, None)
        save_json_atomic(claims, DP_CLAIMS)

if WORKER_MODE:
    _PENDING_SET = set(_dp_pending())                       # fixed for the run (classify_threads + HF-done are frozen)
    def _queue_remaining(): return len(_PENDING_SET - _dp_all_done())   # TRUE shared-queue remaining (counts BOTH workers)
    print(f"[dp-worker {WID}] {_queue_remaining():,} threads remaining in shared queue; claiming {CLAIM_SIZE} at a time", flush=True)
    ensure_llm()
    _n = 0; batch = []; _cum = {}        # _cum persists across claims -> cum rate spans the whole worker session
    try:
        while True:
            batch = _claim_next(CLAIM_SIZE)
            if not batch:
                print(f"[dp-worker {WID}] queue empty — finished. labeled {_n:,} threads this session.", flush=True); break
            try:
                classify_comment_threads(batch, save_every=CLAIM_SIZE, labels_name=DP_LABELS, done_name=DP_DONE,
                                         remaining_fn=_queue_remaining, cum=_cum)
                _n += len(batch)
            finally:
                _release(batch)            # release whether the batch finished or was interrupted mid-way
    except KeyboardInterrupt:
        _release(batch)
        print(f"[dp-worker {WID}] interrupted; released {len(batch)} claimed thread(s). Engine stays loaded — "
              f"re-run THIS cell to resume.", flush=True)
    else:
        print(f"[dp-worker {WID}] done. Merge from the MAIN notebook (WORKER_MODE=False).", flush=True)
        raise SystemExit(f"[dp-worker {WID}] worker finished — later cells are skipped in worker mode (expected).")
else:
    print("[dp-worker] WORKER_MODE=False -> skipping (this notebook runs the single-engine §8c/§9 path below)", flush=True)

[dp-worker] WORKER_MODE=False -> skipping (this notebook runs the single-engine §8c/§9 path below)
CPU times: user 0 ns, sys: 1.66 ms, total: 1.66 ms
Wall time: 1.12 ms


## 8c · Benchmark / timing stage

Times **(a)** the vLLM engine load and **(b)** classifying the next `BENCH_THREADS` unprocessed threads, then
extrapolates a full-run ETA. Benchmarked threads are checkpointed like any others (no wasted work).

In [16]:
%%time
order = sorted(REPROCESS_THREADS)
done_now = _dp_all_done()       # all v5 ledgers (the §9b DP workers' -dp0/-dp1 + main) -> skip finished threads
remaining = [n for n in order if n not in done_now]
n_ran = min(BENCH_THREADS, len(remaining))
print(f"[vllm] {len(set(order) & done_now):,} of {len(order):,} reprocess-threads done | "
      f"{len(remaining):,} remaining -> benchmark runs next {n_ran} (BENCH_THREADS={BENCH_THREADS})", flush=True)

_t = time.time(); ensure_llm(); load_s = time.time() - _t
print(f"[bench] vLLM engine load: {load_s:.1f}s", flush=True)

_t = time.time()
# Bench only GENUINELY-undone threads. `remaining` is already filtered by _dp_all_done() (HF + all vLLM
# ledgers incl. DP workers); passing `order` here would let classify re-filter against only the main ledger
# and re-label worker-done threads (harmless dupes, but wasted GPU). When all done, this is a clean no-op.
_pre_parts = set(ck(COMMENT_LABELS).glob("part_*.parquet")) if ck(COMMENT_LABELS).exists() else set()
classify_comment_threads(remaining, save_every=SAVE_EVERY_THREADS, bench_limit=BENCH_THREADS)
run_s = time.time() - _t
# PARSE COVERAGE of this bench: read only the part-files it just wrote and count model-parsed vs §E-routed vs
# FAILED, breaking the failures down by fail_reason (G) so you see WHY any are unlabeled — want FAILED ~0%.
_newp = [f for f in ck(COMMENT_LABELS).glob("part_*.parquet") if f not in _pre_parts] if ck(COMMENT_LABELS).exists() else []
if _newp:
    _bl = pd.concat([pd.read_parquet(f) for f in _newp], ignore_index=True)
    _pf = int((~_bl["parsed"].astype(bool)).sum())
    _rt = int(_bl["routed"].astype(bool).sum()) if "routed" in _bl.columns else 0
    _model_ok = len(_bl) - _pf - _rt
    _fr = (_bl[~_bl["parsed"].astype(bool)]["fail_reason"].value_counts().to_dict()
           if "fail_reason" in _bl.columns else {})
    print(f"[bench] PARSE COVERAGE: {len(_bl):,} comments this bench | model-parsed {_model_ok:,} | "
          f"§E-routed {_rt:,} | FAILED {_pf:,} ({_pf/max(len(_bl),1):.2%}) by reason {_fr}", flush=True)
    print("[bench]   reasons: empty=blank/[]/{} (model declined), malformed=unparseable JSON, "
          "missing_fields=object lacked about_topic  <- want FAILED ~0%", flush=True)
else:
    print("[bench] PARSE COVERAGE: no new comments labeled this bench (already done)", flush=True)
if n_ran:
    per = run_s / n_ran
    print(f"[bench] {n_ran} threads in {run_s:.0f}s = {per:.2f}s/thread", flush=True)
    print(f"[bench] EXTRAPOLATED {len(remaining):,} threads remaining ~ {per*len(remaining)/3600:.1f} h "
          f"(one-time; cache + checkpoints make reruns free)", flush=True)
else:
    print("[bench] nothing left to benchmark — all threads already processed", flush=True)

[vllm] 49,504 of 49,504 reprocess-threads done | 0 remaining -> benchmark runs next 0 (BENCH_THREADS=500)
[vllm] loading Qwen/Qwen2.5-7B-Instruct-AWQ | TP=1 | gpu_mem_util=0.9 | max_model_len=8192 | CUDA_VISIBLE_DEVICES=1
[vllm] enforce_eager=False (nvcc found)
INFO 06-15 14:26:53 [utils.py:278] non-default args: {'tokenizer': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 128, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}


INFO 06-15 14:26:56 [model.py:617] Resolved architecture: Qwen2ForCausalLM
INFO 06-15 14:26:56 [model.py:1752] Using max model len 8192
INFO 06-15 14:26:58 [awq_marlin.py:268] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 06-15 14:26:58 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files: 100%|██████████| 2/2 [00:01<00:00,  1.29it/s]

INFO 06-15 14:27:00 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-15 14:27:00 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


WARNING 06-15 14:27:04 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: WSL is detected and NVML is not compatible with fork
(EngineCore pid=456534) INFO 06-15 14:27:10 [core.py:112] Initializing a V1 LLM engine (v0.22.1) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=awq_marlin, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache

...

(EngineCore pid=456534) INFO 06-15 14:27:26 [weight_utils.py:922] Filesystem type for checkpoints: EXT4. Checkpoint size: 5.19 GiB. Available RAM: 15.22 GiB.
(EngineCore pid=456534) INFO 06-15 14:27:26 [weight_utils.py:945] Auto-prefetch is disabled because the filesystem (EXT4) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.34s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.49s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.61s/it]
(EngineCore pid=456534) 


(EngineCore pid=456534) INFO 06-15 14:27:30 [default_loader.py:397] Loading weights took 3.23 seconds
(EngineCore pid=456534) INFO 06-15 14:27:32 [gpu_model_runner.py:5132] Model loading took 5.29 GiB memory and 10.930479 seconds
(EngineCore pid=456534) INFO 06-15 14:27:35 [backends.py:1089] Using cache directory: ~/.cache/vllm/torch_compile_cache/eccc93a71b/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=456534) INFO 06-15 14:27:35 [backends.py:1148] Dynamo bytecode transform time: 2.37 s
(EngineCore pid=456534) INFO 06-15 14:27:36 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 0.782 s
(EngineCore pid=456534) INFO 06-15 14:27:36 [decorators.py:311] Directly load AOT compilation from path ~/.cache/vllm/torch_compile_cache/torch_aot_compile/116be48cc72e463ed928e07ae4a2104f9ad7f7dad6fa81bbd2f535aea4daf12d/rank_0_0/model
(EngineCore pid=456534) INFO 06-15 14:27:36 [monitor.py:53] torch.compile took 3.40 s in total
(EngineC

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 35/35 [00:01<00:00, 19.64it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 19/19 [00:00<00:00, 25.41it/s]


(EngineCore pid=456534) INFO 06-15 14:27:52 [gpu_model_runner.py:6456] Graph capturing finished in 12 secs, took 0.00 GiB
(EngineCore pid=456534) INFO 06-15 14:27:52 [gpu_worker.py:619] CUDA graph pool memory: 0.0 GiB (actual), 0.31 GiB (estimated), difference: 0.31 GiB (33764147200.0%).
(EngineCore pid=456534) INFO 06-15 14:27:52 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=456534) INFO 06-15 14:27:52 [core.py:302] init engine (profile, create kv cache, warmup model) took 19.92 s (compilation: 3.40 s)
(EngineCore pid=456534) INFO 06-15 14:27:55 [vllm.py:977] Asynchronous scheduling is enabled.
WARNING 06-15 14:27:55 [interface.py:729] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
(EngineCore pid=456534) INFO 06-15 14:27:55 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])

## 9 · Reprocess v1's omitted comments (resumable)

Re-labels ONLY v1's parse-failed comments (the `REPROCESS_THREADS`), checkpointing every `SAVE_EVERY_THREADS`.
Interrupt and re-run to resume. v1's good labels were already seeded in §8d.

In [17]:
%%time
# Skip every reprocess-thread already done by the §9b workers (-dp0/-dp1) or a prior run -> clean no-op when done.
_done = _dp_all_done()
pending = [t for t in sorted(REPROCESS_THREADS) if t not in _done]
print(f"[C-comments] {len(_done & set(REPROCESS_THREADS)):,} of {len(REPROCESS_THREADS):,} reprocess-threads done; "
      f"vLLM will process {len(pending):,}", flush=True)
classify_comment_threads(pending, save_every=SAVE_EVERY_THREADS)
comments_df = load_vllm_comments()   # v5 = v1-good seed + reprocessed omissions; parsed=False DROPPED
_expected = thr_comments["id"].nunique()
_omitted  = _expected - len(comments_df)
print(f"[C-comments] assembly {_expected:,} | labeled (parsed=True) {len(comments_df):,} | STILL OMITTED {_omitted:,} "
      f"({_omitted/max(_expected,1):.2%}) | about_topic={int(comments_df.about_topic.sum()):,}", flush=True)

[C-comments] 49,504 of 49,504 reprocess-threads done; vLLM will process 0
comment threads: 0 done | 0 to process now
[C-comments] assembly 5,778,767 | labeled (parsed=True) 5,778,767 | STILL OMITTED 0 (0.00%) | about_topic=411,975
CPU times: user 2.51 s, sys: 1.3 s, total: 3.81 s
Wall time: 3.26 s


## 10 · Reprocess v1's omitted posts (resumable)

In [18]:
%%time
anchor_post_names = (set(lex_anchor_posts.select('name').to_series().to_list())
                     | {t3(i) for i in sem_anchor_post_ids}) & classify_threads   # ALL anchored posts (for §11)
ap_all = posts.filter(pl.col("name").is_in(list(anchor_post_names))).select(
        ["id","name","title","selftext_clean","subreddit","hp"]).to_pandas()
# v5: classify ONLY v1's parse-failed posts; v1-good posts were seeded in §8d.
_reproc = ap_all[ap_all["id"].isin(REPROCESS_POST_IDS)].reset_index(drop=True)
print(f"[C-posts] reprocessing {len(_reproc):,} v1-omitted posts (of {len(ap_all):,} anchored)", flush=True)
classify_posts(_reproc, save_every=SAVE_EVERY_POSTS)

_lab = load_parts_all(POST_LABELS).drop_duplicates("id")   # seed (v1-good) + reprocessed
ap = ap_all.merge(_lab, on="id", how="left")
ap["about_topic"] = ap["about_topic"].fillna(ap["hp"]).astype(bool)
ap["stance"]      = ap["stance"].fillna("NA")
ap["confidence"]  = ap["confidence"].fillna(0.0)
print(f"[C-posts] {len(ap):,} posts labeled | about_topic={int(ap.about_topic.sum()):,}", flush=True)

[C-posts] reprocessing 0 v1-omitted posts (of 31,348 anchored)
posts: 0 done | 0 to process now
[C-posts] 31,348 posts labeled | about_topic=3,363
CPU times: user 192 ms, sys: 109 ms, total: 301 ms
Wall time: 81.3 ms


## 11 · Assemble outputs + manifest

Single-process assembly (no shard merge). All outputs carry the `-vllm` suffix so they sit alongside, and never
overwrite, the HF run's `relevant_*.parquet` / `manifest.json`.

In [21]:
%%time
comments_df = load_vllm_comments()   # v5 = v1-good seed + reprocessed omissions (model-parsed/§E-routed), parsed=False dropped
ap = (posts.filter(pl.col("name").is_in(list(anchor_post_names)))
          .select(["id","name","title","selftext_clean","subreddit","hp"]).to_pandas()
          .merge(load_parts_all(POST_LABELS).drop_duplicates("id"), on="id", how="left"))
ap["about_topic"] = ap["about_topic"].fillna(ap["hp"]).astype(bool)
ap["stance"]      = ap["stance"].fillna("NA")
ap["confidence"]  = ap["confidence"].fillna(0.0)

# FULL labeled sets — EVERY processed comment & post (relevant AND off-topic), with metadata. `about_topic` is
# the relevance flag; these are unfiltered so you can slice them yourself. The relevant_* files below are just
# these filtered to about_topic==True.
labeled_comments = comments_df.merge(thr_comments[["id","link_id","author","score","created_utc"]], on="id", how="left")
labeled_comments["thread_tier1"] = labeled_comments["link_id"].isin(tier1_threads)   # comment's thread is Tier-1
save_parquet(labeled_comments, f"labeled_comments{VTAG}.parquet")
labeled_posts = ap.copy()
labeled_posts["matched_tier1"] = labeled_posts["hp"]                                 # this POST matched a Tier-1 term
labeled_posts["thread_tier1"]  = labeled_posts["name"].isin(tier1_threads)           # this post's thread is Tier-1
save_parquet(labeled_posts, f"labeled_posts{VTAG}.parquet")
# Gate used by the sentiment session: keep an item if its thread is Tier-1 OR the LLM marked it relevant.
_gate = labeled_comments["thread_tier1"] | labeled_comments["about_topic"]
print(f"[full] labeled_comments{VTAG}.parquet: {len(labeled_comments):,} rows ({int(labeled_comments.about_topic.sum()):,} "
      f"relevant; {int(_gate.sum()):,} pass Tier-1-or-relevant gate) | labeled_posts{VTAG}.parquet: {len(labeled_posts):,} "
      f"rows ({int(labeled_posts.about_topic.sum()):,} relevant)", flush=True)

rel_posts = ap[ap["about_topic"]].copy(); rel_posts["matched_tier1"] = rel_posts["hp"]
save_parquet(rel_posts, REL_POSTS_OUT)

rel_comments = comments_df[comments_df["about_topic"]].merge(
    thr_comments[["id","link_id","author","score","created_utc"]], on="id", how="left")
save_parquet(rel_comments, REL_COMM_OUT)

manifest = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"), "engine": "vllm", "device": f"CUDA_VISIBLE_DEVICES={VLLM_GPUS}",
    "llm_model": LLM_MODEL, "model_tag": MODEL_TAG, "vtag": VTAG, "llm_revision": LLM_REVISION,
    "prompt_version": PROMPT_VERSION, "tensor_parallel_size": TP_SIZE, "max_model_len": MAX_MODEL_LEN,
    "embed_model": EMBED_MODEL, "embed_sim_threshold": EMBED_SIM_THRESHOLD,
    "anchored_threads": len(anchored_threads),
    "capture_recapture": load_json("capture_recapture.json") or {},
    "n_relevant_posts": int(len(rel_posts)), "n_relevant_comments": int(len(rel_comments)),
    "stance_posts": rel_posts["stance"].value_counts().to_dict(),
    "stance_comments": rel_comments["stance"].value_counts().to_dict(),
}
save_json(manifest, MANIFEST_OUT)
print(json.dumps(manifest, indent=2))

[full] labeled_comments-vllm-v5.parquet: 5,778,767 rows (411,975 relevant; 1,335,440 pass Tier-1-or-relevant gate) | labeled_posts-vllm-v5.parquet: 31,348 rows (3,363 relevant)
{
  "generated_at": "2026-06-15 18:43:19",
  "engine": "vllm",
  "device": "CUDA_VISIBLE_DEVICES=1",
  "llm_model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
  "model_tag": "7B-AWQ",
  "vtag": "-vllm-v5",
  "llm_revision": null,
  "prompt_version": "v5-v1prompt-object",
  "tensor_parallel_size": 1,
  "max_model_len": 8192,
  "embed_model": "all-MiniLM-L6-v2",
  "embed_sim_threshold": 0.45,
  "anchored_threads": 85383,
  "capture_recapture": {
    "n_lexical": 77624,
    "n_semantic": 17640,
    "overlap": 9881,
    "union": 85383,
    "N_hat": 138578,
    "est_recall": 0.6161
  },
  "n_relevant_posts": 3363,
  "n_relevant_comments": 411975,
  "stance_posts": {
    "anti": 1093,
    "pro": 945,
    "neutral": 729,
    "NA": 509,
    "mixed": 87
  },
  "stance_comments": {
    "anti": 160543,
    "pro": 158060,
    "neutral

## 12 · Label-free validation (pseudo-labels)

Tier-1 posts should be ~all `about_topic=True`; a keyword-free deep-tail sample should be ~all `False`.

In [22]:
%%time
t1_ids = set(tier1_posts.select("id").to_series().to_list()) & set(ap["id"])
if t1_ids:
    sub = ap[ap["id"].isin(t1_ids)]
    print(f"Tier-1 posts judged about_topic: {sub.about_topic.mean():.1%} of {len(sub):,} (want ~100%)")

tail = posts.filter(~pl.col("cand")).select(["id","title","selftext_clean"]).sample(
        n=min(60, posts.filter(~pl.col('cand')).height), seed=0).to_pandas()
_tail_rows = list(tail.itertuples())
prompts = [POST_INSTRUCTIONS + "\n\nPOSTS:\n" + f"[{r.id}] {str(r.title)[:200]} :: {str(r.selftext_clean)[:500]}"
           for r in _tail_rows]                                 # A: one post per call -> single object
lab = {}
for r, out in zip(_tail_rows, llm_generate(prompts)):
    _it, _ = parse_item(out); lab[r.id] = _it
flush_cache()
fp = np.mean([bool(lab.get(i,{}).get("about_topic", False)) for i in tail["id"]]) if len(tail) else float("nan")
print(f"Deep-tail posts judged about_topic: {fp:.1%} of {len(tail):,} (want ~0%)")

Tier-1 posts judged about_topic: 80.2% of 3,588 (want ~100%)


Processed prompts:   0%|          | 0/60 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=456534) WARNING 06-15 18:43:23 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=456534) WARNING 06-15 18:43:24 [jit_monitor.py:103] Triton kernel JIT compilation during inference: apply_token_bitmask_inplace_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 60/60 [00:07<00:00,  8.54it/s, est. speed input: 2479.61 toks/s, output: 264.91 toks/s]

Deep-tail posts judged about_topic: 0.0% of 60 (want ~0%)
CPU times: user 640 ms, sys: 326 ms, total: 966 ms
Wall time: 7.49 s


## 13 · Summary

In [23]:
%%time
print("=== COVERAGE (parsed=False are DROPPED, so 'omitted' = comments not successfully labeled) ===")
_exp = thr_comments["id"].nunique()
_om  = _exp - len(comments_df)
print(f"comments: assembly {_exp:,} | labeled {len(comments_df):,} | STILL OMITTED {_om:,} "
      f"({_om/max(_exp,1):.2%})   <- want this ~0")

# G: of the LABELED comments, how many were model-parsed vs §E-routed; and of every UNLABELED (parsed=False)
# row written, break the failures down by fail_reason so you see WHY they're missing (empty/malformed/missing).
if "routed" in comments_df.columns:
    _rt = int(comments_df["routed"].astype(bool).sum())
    print(f"  labeled breakdown: model-parsed {len(comments_df)-_rt:,} | §E-routed (no-signal) {_rt:,}")
_allc = load_parts_all(f"comment_labels{VTAG}")
if "fail_reason" in _allc.columns:
    _fail = _allc[~_allc["parsed"].astype(bool)]
    if len(_fail):
        print(f"  parse FAILURES ({len(_fail):,} rows) by reason: {_fail['fail_reason'].value_counts().to_dict()}")
        print("    empty=blank/[]/{} (model declined) | malformed=unparseable JSON | missing_fields=no about_topic")
    else:
        print("  parse FAILURES: 0 — every processed comment yielded a usable object or was §E-routed")
print(f"\n=== RELEVANCE (of the labeled set) ===")
print(f"comments: {int(comments_df.about_topic.sum()):,} relevant of {len(comments_df):,} ({comments_df.about_topic.mean():.1%})")
print(f"posts:    {len(ap):,} labeled | {int(ap.about_topic.sum()):,} relevant ({ap.about_topic.mean():.1%})\n")

print("=== RELEVANT POSTS (vLLM) ===", len(rel_posts))
print(rel_posts["stance"].value_counts().to_string())
print("\n=== RELEVANT COMMENTS (vLLM) ===", len(rel_comments))
print(rel_comments["stance"].value_counts().to_string())
print("\nrefers_to_parent (context-resolved):",
      f"{rel_comments['refers_to_parent'].mean():.1%}" if len(rel_comments) else "n/a")
print("\nSample relevant comments:")
for r in rel_comments.sample(min(10, len(rel_comments)), random_state=0).itertuples():
    print(f"  [{r.stance:7}] ref={r.refers_to_parent}  {str(thr_comments.set_index('id').loc[r.id,'body'])[:90]}")

=== COVERAGE (parsed=False are DROPPED, so 'omitted' = comments not successfully labeled) ===
comments: assembly 5,778,767 | labeled 5,778,767 | STILL OMITTED 0 (0.00%)   <- want this ~0
  labeled breakdown: model-parsed 5,705,223 | §E-routed (no-signal) 73,544
  parse FAILURES: 0 — every processed comment yielded a usable object or was §E-routed

=== RELEVANCE (of the labeled set) ===
comments: 411,975 relevant of 5,778,767 (7.1%)
posts:    31,348 labeled | 3,363 relevant (10.7%)

=== RELEVANT POSTS (vLLM) === 3363
stance
anti       1093
pro         945
neutral     729
NA          509
mixed        87

=== RELEVANT COMMENTS (vLLM) === 411975
stance
anti       160543
pro        158060
neutral     87824
mixed        5211
NA            337

refers_to_parent (context-resolved): 37.7%

Sample relevant comments:
  [anti   ] ref=True  You don't think I'm aware, fucko? You haven't won. It's still happening.
  [anti   ] ref=False  What does that have to do with drunk driving or this article?
  